### Chains and Tool Calling Using LangGraph

In this section, we will explore how to build a tool-calling chain using LangGraph covering 4 core concepts:
1. **Chat Messages as Graph State**: Capturing multi-turn conversations using message objects.
2. **Chat Models in Graph Nodes**: Integrating LLMs inside graph nodes.
3. **Binding Tools to Chat Models**: Providing tools (`@tool` / Python functions) to the model.
4. **Executing Tool Calls in Nodes**: Using `ToolNode` and `tools_condition` to route and execute tool calls.


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")


#### 1. Messages in LangChain & LangGraph
Messages capture different roles within a conversation:
- `HumanMessage`: Message from the human user.
- `AIMessage`: Response message from the chat model.
- `SystemMessage`: Instructions setting the behavior of the chat model.
- `ToolMessage`: The output resulting from a tool execution.

Every message contains:
- `content`: Text content or multimodal data.
- `name`: (Optional) Name of the author.
- `response_metadata` / `tool_calls`: Metadata populated by model providers (e.g. tool call arguments).


In [1]:
from langchain_core.messages import AIMessage, HumanMessage

messages = [
    AIMessage(content="Please tell me how can I help you today.", name="Assistant"),
    HumanMessage(content="I want to learn coding", name="Krish"),
    AIMessage(content="Which programming language do you want to learn?", name="Assistant"),
    HumanMessage(content="I want to learn Python programming language.", name="Krish")
]

for message in messages:
    message.pretty_print()


================================== Ai Message ==================================
Name: Assistant

Please tell me how can I help you today.
================================ Human Message =================================
Name: Krish

I want to learn coding
================================== Ai Message ==================================
Name: Assistant

Which programming language do you want to learn?
================================ Human Message =================================
Name: Krish

I want to learn Python programming language.


#### 2. Chat Models
We can invoke chat models (such as Groq's fast LLMs or OpenAI) with a sequence of messages.


In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")
# Or with OpenAI:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini")

result = llm.invoke(messages)
print("LLM Response:")
print(result.content)


NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [ ]:
print("Response Metadata:")
print(result.response_metadata)


#### 3. Defining and Binding Tools
Tools allow LLMs to interact with external systems, APIs, or custom functions.

We can define a tool using the `@tool` decorator from `langchain_core.tools` with type hints and docstrings.


In [ ]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two integers a and b.

    Args:
        a (int): First integer.
        b (int): Second integer.

    Returns:
        int: The sum of a and b.
    """
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers a and b.

    Args:
        a (int): First integer.
        b (int): Second integer.

    Returns:
        int: The product of a and b.
    """
    return a * b

tools = [add, multiply]


In [ ]:
### Binding tools with the LLM
llm_with_tools = llm.bind_tools(tools)

tool_call_response = llm_with_tools.invoke([HumanMessage(content="What is 2 plus 2?", name="Krish")])
print("AI Response content:", tool_call_response.content)
print("Tool Calls generated:", tool_call_response.tool_calls)


#### 4. Using Messages as State & Reducers

In LangGraph, if no reducer function is specified, state updates overwrite the previous value.

To append new messages to the conversation history instead of replacing it, we annotate the list with the `add_messages` reducer.


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


##### How `add_messages` works
`add_messages` is the helper function that LangGraph runs behind the scenes when updating state messages.
Below, we can inspect how `add_messages` combines an existing message list with a new message:


In [ ]:
initial_messages = [
    AIMessage(content="Please tell me how can I help", name="Assistant"),
    HumanMessage(content="I want to learn coding", name="Krish")
]
ai_reply = AIMessage(content="Which programming language would you like to start with?", name="Assistant")

updated_messages = add_messages(initial_messages, ai_reply)
for msg in updated_messages:
    msg.pretty_print()


##### ⚠️ Crucial Concept: Why Reducers Matter Inside `StateGraph` (With vs Without Reducer)

When calling `add_messages(initial_messages, ai_reply)` directly in Python, it simply acts as a helper function to append messages.

**The real impact of `Annotated[..., add_messages]` is inside LangGraph's execution engine:**
- **Without Reducer (`messages: list[AnyMessage]`):** When a node returns `{"messages": [new_msg]}`, LangGraph **overwrites** the existing state. All previous conversation history is lost!
- **With Reducer (`messages: Annotated[list[AnyMessage], add_messages]`):** When a node returns `{"messages": [new_msg]}`, LangGraph automatically runs `add_messages` behind the scenes to **append** the new message to existing history.

Let's verify this behavior side-by-side with a live graph example:


In [ ]:
from langgraph.graph import StateGraph, START, END

# 1. State WITHOUT Reducer (Overwrites History)
class StateNoReducer(TypedDict):
    messages: list[AnyMessage]

def bot_no_reducer(state: StateNoReducer) -> dict:
    return {"messages": [AIMessage(content="I am fine!")]}

builder_no = StateGraph(StateNoReducer)
builder_no.add_node("bot", bot_no_reducer)
builder_no.add_edge(START, "bot")
builder_no.add_edge("bot", END)
graph_no = builder_no.compile()

test_input = {"messages": [HumanMessage(content="Hi"), HumanMessage(content="How are you?")]}
res_no = graph_no.invoke(test_input)
print("--- WITHOUT Reducer ---")
print(f"Total messages in state: {len(res_no['messages'])} (History Overwritten! Only new reply kept)")
for m in res_no["messages"]:
    print(f"  - {m.content}")

# 2. State WITH Reducer (Preserves and Appends History)
class StateWithReducer(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def bot_with_reducer(state: StateWithReducer) -> dict:
    return {"messages": [AIMessage(content="I am fine!")]}

builder_with = StateGraph(StateWithReducer)
builder_with.add_node("bot", bot_with_reducer)
builder_with.add_edge(START, "bot")
builder_with.add_edge("bot", END)
graph_with = builder_with.compile()

res_with = graph_with.invoke(test_input)
print("\n--- WITH Reducer ---")
print(f"Total messages in state: {len(res_with['messages'])} (Full history preserved!)")
for m in res_with["messages"]:
    print(f"  - {m.content}")


#### Building the Tool-Calling Graph

We now create:
1. **`llm_tool` Node**: Calls `llm_with_tools.invoke(state["messages"])`.
2. **`ToolNode`**: Prebuilt node that executes any tool calls generated by the LLM.
3. **`tools_condition`**: Prebuilt conditional edge that checks if the latest message has `tool_calls`. If yes, it routes to `tools`; if not, it routes to `END`.
4. **ReAct edge**: From `tools`, we connect back to `llm_tool` so the LLM receives the tool results and generates a final natural language answer for the user.


In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

## Node definition
def llm_tool(state: State) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

## Build Graph
builder = StateGraph(State)

builder.add_node("llm_tool", llm_tool)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "llm_tool")
builder.add_conditional_edges(
    "llm_tool",
    # If LLM generated tool_calls -> route to "tools"
    # If LLM generated normal text -> route to END
    tools_condition
)
# Loop back to llm_tool to synthesize the tool response
builder.add_edge("tools", "llm_tool")

graph = builder.compile()

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())


#### Graph Invocation: Tool Calling Query
Let's ask a question that triggers tool execution (`add`).


In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="What is 2 plus 2?")]})

print("\n--- Full Message History ---")
for message in result["messages"]:
    message.pretty_print()


#### Graph Invocation: Direct Conversation Query
Let's ask a general question that does not require tools.


In [ ]:
result_general = graph.invoke({"messages": [HumanMessage(content="What is Machine Learning?")]})

print("\n--- Full Message History ---")
for message in result_general["messages"]:
    message.pretty_print()
